# 01 — Exploration GTFS-RT STM
**Phase 1 / Semaine 1**

Objectifs de ce notebook :
1. Se connecter à l'API GTFS-RT de la STM
2. Afficher les positions des véhicules en temps réel sur une carte Folium
3. Explorer la structure des TripUpdates (délais)
4. Comprendre le format GTFS statique (stops.txt, routes.txt)

**Prérequis :**
- Clé API STM dans votre `.env`
- `pip install -r requirements.txt`

In [ ]:
import sys
sys.path.insert(0, '..')   # Pour importer les modules src/

from dotenv import load_dotenv
load_dotenv('../.env')

print('Environnement chargé ✓')

## 1. Récupérer les positions des véhicules

In [ ]:
from src.collector.gtfs_client import GTFSClient

client = GTFSClient()
positions = client.get_vehicle_positions()

print(f'Nombre de véhicules actifs : {len(positions)}')
print(f'\nExemple de position :')
print(positions[0] if positions else 'Aucune position reçue')

## 2. Visualiser sur une carte Folium

Folium génère une carte Leaflet interactive directement dans le notebook.

In [ ]:
import folium
import pandas as pd

# Centre sur Montréal
m = folium.Map(location=[45.5017, -73.5673], zoom_start=12, tiles='CartoDB positron')

# Ajouter un marqueur par véhicule
for pos in positions:
    folium.CircleMarker(
        location=[pos.latitude, pos.longitude],
        radius=4,
        color='#E63946',
        fill=True,
        fill_opacity=0.8,
        popup=f'Ligne {pos.route_id} | Véhicule {pos.vehicle_id}',
        tooltip=f'Route {pos.route_id}',
    ).add_to(m)

print(f'{len(positions)} véhicules affichés')
m   # Affiche la carte dans le notebook

## 3. Explorer les TripUpdates (délais)

TODO : explorer les délais par ligne, distribution des retards, etc.

In [ ]:
updates = client.get_trip_updates()

# Aplatir en DataFrame pour l'exploration
rows = []
for u in updates:
    for stu in u.stop_updates:
        rows.append({
            'route_id': u.route_id,
            'trip_id': u.trip_id,
            'stop_id': stu.stop_id,
            'arrival_delay': stu.arrival_delay,
        })

df = pd.DataFrame(rows)
print(f'Shape : {df.shape}')
df.head(10)

In [ ]:
import matplotlib.pyplot as plt

# Distribution des délais d'arrivée
delays = df['arrival_delay'].dropna()
delays_clipped = delays.clip(-300, 600)  # On garde -5min à +10min

plt.figure(figsize=(10, 4))
plt.hist(delays_clipped, bins=50, color='#E63946', alpha=0.8, edgecolor='white')
plt.xlabel('Délai d\'arrivée (secondes)')
plt.ylabel('Nombre d\'arrêts')
plt.title('Distribution des délais STM (snapshot actuel)')
plt.axvline(0, color='black', linestyle='--', linewidth=1, label='À l\'heure')
plt.legend()
plt.tight_layout()
plt.show()

print(f'\nDélai médian : {delays.median():.0f}s')
print(f'Délai moyen  : {delays.mean():.0f}s')
print(f'% en retard  : {(delays > 0).mean()*100:.1f}%')

## 4. Prochaines étapes

- [ ] Télécharger le GTFS statique STM (https://www.stm.info/fr/a-propos/developpeurs)
- [ ] Charger `stops.txt` et `routes.txt` dans PostGIS
- [ ] Démarrer `docker-compose up db` et vérifier la connexion avec `src/utils/db.py`
- [ ] Passer au notebook `02_pipeline_ingestion.ipynb`